# 07 · Leave-one-slide-out evaluation and reporting

Read eligible slides, run fixed-budget leave-out folds without checkpoint selection on omitted labels, and save fold results.

## 1. Read the input data

Edit the real input paths and method parameters below. Run the cells in order using the `spagc` kernel. Each notebook reads its own inputs; it does not execute another notebook.

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import torch

ROOT = Path.cwd().resolve()
while not (ROOT / "notebooks/02_he_prediction").is_dir():
    if ROOT == ROOT.parent:
        raise FileNotFoundError("Start Jupyter inside the spaGC repository.")
    ROOT = ROOT.parent


if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
RUN_ID = os.environ.get("SPAGC_RUN_ID", "he_reference_v2")
OUT = ROOT / "outputs/he_prediction" / RUN_ID
DEVICE = os.environ.get(
    "SPAGC_DEVICE", "cuda:0" if torch.cuda.is_available() else "cpu"
)
FEATURE_DIR = OUT / "features"
SLIDE_ROWS = [
    ("proseg1", 984, 1784, 27761, 50000, "proseg1.tif", "proseg1.h5"),
    ("proseg2", 824, 1784, 23140, 50000, "proseg2.tif", "proseg2.h5"),
    ("proseg3", 856, 1784, 23981, 50000, "proseg3.tif", "proseg3.h5"),
    ("proseg4", 912, 1784, 25667, 50000, "proseg4.tif", "proseg4.h5"),
    ("proseg5", 896, 1784, 25108, 50000, "proseg5.tif", "proseg5.h5"),
    ("proseg6", 912, 1784, 25706, 50000, "proseg6.tif", "proseg6.h5"),
    ("proseg7", 1624, 3472, 45603, 97266, "proseg7.tif", "proseg7.h5"),
    ("proseg8", 1480, 3472, 41510, 97267, "proseg8.tif", "proseg8.h5"),
    ("proseg9", 1624, 3472, 45603, 97262, "proseg_9.tif", "proseg_9.h5"),
    ("proseg10", 1624, 3472, 45605, 97259, "proseg_10.tif", "proseg_10.h5"),
]
SLIDES = []
for (
    name,
    height,
    width,
    native_height,
    native_width,
    image_name,
    feature_name,
) in SLIDE_ROWS:
    index = name.removeprefix("proseg")
    SLIDES.append(
        {
            "name": name,
            "h5ad_path": str(
                ROOT / "data/he_aligned_h5ad" / f"proseg_{index}_aligned.h5ad"
            ),
            "h5_path": str(FEATURE_DIR / feature_name),
            "he_tif_path": str(ROOT / "data/he_slides" / image_name),
            "niche_label_path": str(
                ROOT
                / "data/he_export_2026_10_09/labels"
                / f"Proseg{index}"
                / "label_niche_uint16.dat"
            ),
            "domain_label_path": str(
                ROOT
                / "data/he_export_2026_10_09/labels"
                / f"Proseg{index}"
                / "label_domain_uint16.dat"
            ),
            "label_shape": [height, width],
            "H": height,
            "W": width,
            "level0_shape": [native_height, native_width],
            "he_shape": [height * 28, width * 28],
            "region_size": 28,
        }
    )
TRAIN_SLIDES = SLIDES[:8]
VALIDATION_SLIDES = SLIDES[8:]
CFG = {
    "model": {
        "num_domain_classes": 5,
        "num_niche_classes": 21,
        "ctrans_in_dim": 768,
        "ctrans_dim": 256,
        "rgb_dim": 32,
        "refine_depth": 2,
        "region_grid": 8,
        "region_size": 28,
        "dropout": 0.1,
    },
    "data": {
        "tile_size": 224,
        "region_size": 28,
        "region_grid": 8,
        "ignore_idx": 0,
        "normalize_rgb": True,
        "load_h5_RAM": "Auto",
        "num_workers": 0,
    },
    "train": {
        "epochs": 100,
        "batch_size": 96,
        "lr": 0.0001,
        "weight_decay": 0.05,
        "max_grad_norm": 1.0,
        "use_amp": True,
        "val_interval": 2,
        "save_interval": 5,
        "loss_weights": {"domain": 1.0, "niche": 1.0},
        "standardize_features": True,
        "augment": True,
        "class_balanced_loss": True,
        "slide_balanced_sampler": True,
        "use_focal": False,
        "focal_gamma": 2.0,
        "lr_schedule": "constant",
        "warmup_epochs": 3,
        "feature_noise_std": 0.05,
        "rgb_jitter": 0.1,
    },
}
SAMPLE_METADATA = os.environ.get("SPAGC_SAMPLE_METADATA")
FOLD_DIR = OUT / "loocv_fixed_epoch"

In [ ]:
from spagc._io import require_files
from spagc.histology.metadata import (
    attach_sample_metadata,
    prepare_slides,
    setup_slide_shapes,
)
from spagc.histology.training import leave_one_slide_out

Read all eligible real slides and fix the training budget before any fold. Patient metadata, if supplied, groups all slides from one patient together.

In [ ]:
TRAIN_SLIDES = attach_sample_metadata(TRAIN_SLIDES, SAMPLE_METADATA)
VALIDATION_SLIDES = attach_sample_metadata(VALIDATION_SLIDES, SAMPLE_METADATA)
slides = prepare_slides(TRAIN_SLIDES + VALIDATION_SLIDES, root=ROOT)
setup_slide_shapes(slides)
options = {
    **CFG["train"],
    "ignore_idx": CFG["data"]["ignore_idx"],
    "load_h5_RAM": CFG["data"]["load_h5_RAM"],
    "num_workers": CFG["data"]["num_workers"],
    "device": DEVICE,
    "num_domain_classes": CFG["model"]["num_domain_classes"],
    "num_niche_classes": CFG["model"]["num_niche_classes"],
    "dropout": CFG["model"]["dropout"],
    "model_config": CFG["model"],
}
options["checkpoint_selection"] = "final"
FOLD_DIR.mkdir(parents=True, exist_ok=True)
for slide in slides:
    require_files(
        slide["h5_path"],
        slide["he_tif_path"],
        slide["niche_label_path"],
        slide["domain_label_path"],
    )

## 2. Fit each omitted-slide or omitted-patient fold

Use a fresh model per fold, save its final checkpoint after the fixed epoch budget, and only then evaluate the omitted data once. Omitted labels do not select checkpoints.

If patient identities are supplied, reject overlap. Without identities, retain slide-level evaluation and report patient independence as unverified.

Train on the declared slides. Ordinary training selects a checkpoint using validation niche mIoU; fixed-epoch leave-out training saves its final checkpoint before evaluation.

Train a fresh model per omitted slide or supplied patient group. Omitted labels never select checkpoints or training epochs.

In [ ]:
summary = leave_one_slide_out(slides, str(FOLD_DIR), **options)
require_files(FOLD_DIR / "summary.json")
assert len(summary) == len({s.get("patient_id", s["name"]) for s in slides})
for name, metrics in summary.items():
    require_files(
        FOLD_DIR / name / "model_final.pth", FOLD_DIR / name / "held_out_metrics.json"
    )
    assert metrics["epoch"] == options["epochs"]
    assert metrics["checkpoint_selection"] == "final"
    assert metrics["held_out_evaluated_after_checkpoint_save"]
    assert not (FOLD_DIR / name / "model_best.pth").exists()
print("Completed fixed-epoch folds:", sorted(summary))

## 3. Inspect the stored fold results

The fold runner saves summary.json, final checkpoints and held-out metrics. Check that all folds used the declared epoch budget and never wrote a best-validation checkpoint.

In [ ]:
print(
    {
        "folds": len(summary),
        "epochs": options["epochs"],
        "summary": str(FOLD_DIR / "summary.json"),
    }
)